In [ ]:
import os
import glob
import multiprocessing as mp

import pandas as pd
from tqdm import tqdm

from analyze_traces import analyze

YELLOW = "\033[33m"
BOLD = "\033[1m"
RESET = "\033[0m"

script_dir = os.getcwd()
plot_dir = os.path.join(script_dir, "..", "plots")
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)

trace_files = glob.glob(os.path.join(script_dir, "input*.jsonl"))
trace_files.sort()

results = []

if trace_files:
    worker_count = min(len(trace_files), mp.cpu_count() or 1)
    if worker_count > 1:
        ctx = mp.get_context("spawn")
        chunksize = max(1, len(trace_files) // (worker_count * 4))
        with ctx.Pool(worker_count) as pool:
            for bundles in tqdm(
                pool.imap_unordered(analyze, trace_files, chunksize=chunksize),
                total=len(trace_files),
                desc="Analyzing",
            ):
                results.extend(bundles)
    else:
        for bundles in tqdm(map(analyze, trace_files), total=len(trace_files), desc="Analyzing"):
            results.extend(bundles)

df = pd.DataFrame(results)
df = df[~df["warmup"]].reset_index(drop=True)

In [ ]:
def safename(name: str) -> str:
    return name.replace('.', '_').replace('%', 'pct')

In [ ]:
import numpy as np

metric_preferences = {'gen_throughput': 'max', 'vsr': 'max', 'latency': 'min'}


def report_minedraft_improvement(metrics_slice: pd.DataFrame, targets: list[str], context: str):
    base_slice = metrics_slice[~metrics_slice['method'].str.startswith('MineDraft')]
    std_slice = metrics_slice[metrics_slice['method'].str.startswith('Standard SD')]
    mine_slice = metrics_slice[
        metrics_slice['method'].str.startswith('MineDraft')
        & ~metrics_slice['method'].str.contains(r'\+')
    ]
    if std_slice.empty:
        std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE')]
        mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE')]
    if std_slice.empty:
        std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE-3')]
        mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE-3')]

    if base_slice.empty or std_slice.empty or mine_slice.empty:
        return

    std_sd = std_slice.groupby('k').first().dropna()
    if std_sd.empty:
        return

    # 1) Max improvement over Std SD across ks
    max_improvements = {target: -np.inf for target in targets}

    # 2) Improvement at the k where baseline is best
    mine_at_base_k = {
        target: (-np.inf if metric_preferences[target] == 'max' else np.inf)
        for target in targets
    }
    base_best_info = {}

    # Compute optimal baseline value at each k: for each target, aggregate across
    # all baseline methods by taking max (if higher-is-better) or min (if lower-is-better)
    baseline_agg = {}
    for target in targets:
        direction = metric_preferences[target]
        col = f'avg_{target}'
        if direction == 'max':
            baseline_agg[col] = base_slice.groupby('k')[col].max()
        else:
            baseline_agg[col] = base_slice.groupby('k')[col].min()
    baseline = pd.DataFrame(baseline_agg).dropna()
    if baseline.empty:
        return

    # Determine baseline best k per target
    for target in targets:
        direction = metric_preferences[target]
        base_series = baseline[f'avg_{target}'].dropna()
        if base_series.empty:
            continue
        k_star = base_series.idxmax() if direction == 'max' else base_series.idxmin()
        base_best_val = base_series.loc[k_star]
        if base_best_val == 0 or not np.isfinite(base_best_val):
            continue
        base_best_info[target] = (k_star, base_best_val)

    # Iterate MineDraft variants
    for _, mine_indexes in mine_slice.groupby('method').groups.items():
        mine_grouped = mine_slice.loc[mine_indexes].groupby('k').first().dropna()
        if len(mine_indexes) != len(mine_grouped) or len(mine_grouped) == 0:
            print(f"{BOLD}{YELLOW}[WARNING]{RESET} NaN or duplicates found in MineDraft results for {context}")

        common_k = std_sd.index.intersection(mine_grouped.index)
        if common_k.empty:
            continue

        for target in targets:
            direction = metric_preferences[target]
            # Max improvement over Std SD across k
            comparison = (
                pd.DataFrame({
                    'mine': mine_grouped.loc[common_k, f'avg_{target}'],
                    'std': std_sd.loc[common_k, f'avg_{target}'],
                })
                .dropna()
            )
            comparison = comparison[comparison['std'] != 0]
            if not comparison.empty:
                if direction == 'max':
                    improvements = (comparison['mine'] - comparison['std']) / comparison['std'] * 100
                else:
                    improvements = (comparison['std'] - comparison['mine']) / comparison['std'] * 100
                if not improvements.empty:
                    max_improvements[target] = max(max_improvements[target], improvements.max())

            # MineDraft value at baseline's best k
            if target in base_best_info:
                k_star, _ = base_best_info[target]
                if k_star in mine_grouped.index:
                    v = mine_grouped.loc[k_star, f'avg_{target}']
                    if np.isfinite(v):
                        if direction == 'max':
                            mine_at_base_k[target] = max(mine_at_base_k[target], v)
                        else:
                            mine_at_base_k[target] = min(mine_at_base_k[target], v)

    # Print per-target report
    for target in targets:
        if target not in base_best_info or not np.isfinite(max_improvements[target]):
            continue
        k_star, base_best_val = base_best_info[target]
        mine_val = mine_at_base_k[target]
        if not np.isfinite(mine_val) or base_best_val == 0:
            continue

        direction = metric_preferences[target]
        if direction == 'max':
            improvement_over_best_base = (mine_val - base_best_val) / base_best_val * 100
        else:
            improvement_over_best_base = (base_best_val - mine_val) / base_best_val * 100

        print(
            f"{context} [{target}] (Base-best@k={k_star}): " +
            (f"max ΔStdSD={max_improvements[target]:.2f}% | " if max_improvements[target] > 0 else f"{BOLD}{YELLOW}max ΔStdSD={max_improvements[target]:.2f}%{RESET} | ") +
            (f"ΔbestBaseline={improvement_over_best_base:.2f}%" if improvement_over_best_base > 0 else f"{BOLD}{YELLOW}ΔbestBaseline={improvement_over_best_base:.2f}%{RESET}")
        )

def report_minedraft_improvement_vs_e(metrics_slice: pd.DataFrame, targets: list[str], context: str):
    std_slice = metrics_slice[metrics_slice['method'].str.startswith('Standard SD')]
    mine_slice = metrics_slice[
        metrics_slice['method'].str.startswith('MineDraft')
        & ~metrics_slice['method'].str.contains(r'\+')
    ]
    if std_slice.empty:
        std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE')]
        mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE')]
    if std_slice.empty:
        std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE-3')]
        mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE-3')]

    if std_slice.empty or mine_slice.empty:
        return

    std_sd = std_slice.groupby('e').first().dropna()
    if std_sd.empty:
        return

    # 1) Max improvement over Std SD across es
    max_improvements = {target: -np.inf for target in targets}

    # Iterate MineDraft variants
    for _, mine_indexes in mine_slice.groupby('method').groups.items():
        mine_grouped = mine_slice.loc[mine_indexes].groupby('e').first().dropna()
        if len(mine_indexes) != len(mine_grouped) or len(mine_grouped) == 0:
            print(f"{BOLD}{YELLOW}[WARNING]{RESET} NaN or duplicates found in MineDraft results for {context}")

        common_e = std_sd.index.intersection(mine_grouped.index)
        if common_e.empty:
            continue

        for target in targets:
            direction = metric_preferences[target]
            # Max improvement over Std SD across e
            comparison = (
                pd.DataFrame({
                    'mine': mine_grouped.loc[common_e, f'avg_{target}'],
                    'std': std_sd.loc[common_e, f'avg_{target}'],
                })
                .dropna()
            )
            comparison = comparison[comparison['std'] != 0]
            if not comparison.empty:
                if direction == 'max':
                    improvements = (comparison['mine'] - comparison['std']) / comparison['std'] * 100
                else:
                    improvements = (comparison['std'] - comparison['mine']) / comparison['std'] * 100
                if not improvements.empty:
                    max_improvements[target] = max(max_improvements[target], improvements.max())

    # Print per-target report
    for target in targets:
        print(
            f"{context} [{target}] : " +
            (f"max ΔStdSD={max_improvements[target]:.2f}%" if max_improvements[target] > 0 else f"{BOLD}{YELLOW}max ΔStdSD={max_improvements[target]:.2f}%{RESET}")
        )


def report_minedraft_improvement_by(metrics_df: pd.DataFrame, targets: list[str], group_by_col: str, context: str):
    if group_by_col not in metrics_df.columns:
        print(f"{BOLD}{YELLOW}[ERROR]{RESET} Column '{group_by_col}' not found in DataFrame")
        return

    unique_groups = metrics_df[group_by_col].unique()

    for group_val in sorted(unique_groups):
        metrics_slice = metrics_df[metrics_df[group_by_col] == group_val]

        base_slice = metrics_slice[~metrics_slice['method'].str.startswith('MineDraft')]
        std_slice = metrics_slice[metrics_slice['method'].str.startswith('Standard SD')]
        mine_slice = metrics_slice[
            metrics_slice['method'].str.startswith('MineDraft')
            & ~metrics_slice['method'].str.contains(r'\+')
        ]
        if std_slice.empty:
            std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE')]
            mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE')]
        if std_slice.empty:
            std_slice = metrics_slice[metrics_slice['method'].str.startswith('EAGLE-3')]
            mine_slice = metrics_slice[metrics_slice['method'].str.startswith('MineDraft + EAGLE-3')]

        if base_slice.empty or std_slice.empty or mine_slice.empty:
            continue

        std_sd = std_slice.groupby('k').first().dropna()
        if std_sd.empty:
            return

        # 1) Max improvement over Std SD across ks
        max_improvements = {target: -np.inf for target in targets}

        # 2) Improvement at the k where baseline is best
        mine_at_base_k = {
            target: (-np.inf if metric_preferences[target] == 'max' else np.inf)
            for target in targets
        }
        base_best_info = {}

        # Compute optimal baseline value at each k: for each target, aggregate across
        # all baseline methods by taking max (if higher-is-better) or min (if lower-is-better)
        baseline_agg = {}
        for target in targets:
            direction = metric_preferences[target]
            col = f'avg_{target}'
            if direction == 'max':
                baseline_agg[col] = base_slice.groupby('k')[col].max()
            else:
                baseline_agg[col] = base_slice.groupby('k')[col].min()
        baseline = pd.DataFrame(baseline_agg).dropna()
        if baseline.empty:
            return

        # Determine baseline best k per target
        for target in targets:
            direction = metric_preferences[target]
            base_series = baseline[f'avg_{target}'].dropna()
            if base_series.empty:
                continue
            k_star = base_series.idxmax() if direction == 'max' else base_series.idxmin()
            base_best_val = base_series.loc[k_star]
            if base_best_val == 0 or not np.isfinite(base_best_val):
                continue
            base_best_info[target] = (k_star, base_best_val)

        # Iterate MineDraft variants
        for _, mine_indexes in mine_slice.groupby('method').groups.items():
            mine_grouped = mine_slice.loc[mine_indexes].groupby('k').first().dropna()
            if len(mine_indexes) != len(mine_grouped) or len(mine_grouped) == 0:
                print(f"{BOLD}{YELLOW}[WARNING]{RESET} {context}: NaN or duplicates found in MineDraft results for {group_by_col}={group_val}")
            
            common_k = std_sd.index.intersection(mine_grouped.index)
            if common_k.empty:
                continue

            for target in targets:
                direction = metric_preferences[target]
                # Max improvement over Std SD across k
                comparison = (
                    pd.DataFrame({
                        'mine': mine_grouped.loc[common_k, f'avg_{target}'],
                        'std': std_sd.loc[common_k, f'avg_{target}'],
                    })
                    .dropna()
                )
                comparison = comparison[comparison['std'] != 0]
                if not comparison.empty:
                    if direction == 'max':
                        improvements = (comparison['mine'] - comparison['std']) / comparison['std'] * 100
                    else:
                        improvements = (comparison['std'] - comparison['mine']) / comparison['std'] * 100
                    if not improvements.empty:
                        max_improvements[target] = max(max_improvements[target], improvements.max())

                # MineDraft value at Std SD's best k
                if target in base_best_info:
                    k_star, _ = base_best_info[target]
                    if k_star in mine_grouped.index:
                        v = mine_grouped.loc[k_star, f'avg_{target}']
                        if np.isfinite(v):
                            if direction == 'max':
                                mine_at_base_k[target] = max(mine_at_base_k[target], v)
                            else:
                                mine_at_base_k[target] = min(mine_at_base_k[target], v)

        # Print per-target report
        for target in targets:
            if target not in base_best_info or not np.isfinite(max_improvements[target]):
                continue
            k_star, base_best_val = base_best_info[target]
            mine_val = mine_at_base_k[target]
            if not np.isfinite(mine_val) or base_best_val == 0:
                continue

            direction = metric_preferences[target]
            if direction == 'max':
                improvement_over_best_base = (mine_val - base_best_val) / base_best_val * 100
            else:
                improvement_over_best_base = (base_best_val - mine_val) / base_best_val * 100

            print(
                f"{context} {group_by_col}={group_val} [{target}] (Std-best@k={k_star}): " +
                (f"max ΔStdSD={max_improvements[target]:.2f}% | " if max_improvements[target] > 0 else f"{BOLD}{YELLOW}max ΔStdSD={max_improvements[target]:.2f}%{RESET} | ") +
                (f"ΔbestBaseline={improvement_over_best_base:.2f}%" if improvement_over_best_base > 0 else f"{BOLD}{YELLOW}ΔbestBaseline={improvement_over_best_base:.2f}%{RESET}")
            )

### n = 1 & default c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

# Index e=0 data (without TETRIS)
no_tetris_bundles = {}
for setting_group, setting_group_indexes in df[(df['e'] == 0) & (df['n'] == 1)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs"]).groups.items():

    dataset_bundles = {}
    for dataset_name, dataset_indexes in df.loc[setting_group_indexes].groupby("dataset").groups.items():
        dataset_bundles[dataset_name] = dataset_indexes

    no_tetris_bundles[setting_group] = dataset_bundles


for (target_model, draft_model, batch_size, reqs), setting_indexes in df[(df['e'] > 0) & (df['n'] == 1) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, e), method_indexes in df.loc[dataset_indexes].groupby(["method", "e"]).groups.items():
            # if "pearl" in method_name or e > 3 or method_name.startswith("tetris") and e > 2:
            #     continue

            method = {"name": method_labels[method_name] + f' (extra={e})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}): "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Find matching e=0 data
        setting_key = (target_model, draft_model, batch_size, reqs)
        dataset_key = dataset_name
        if setting_key in no_tetris_bundles and dataset_key in no_tetris_bundles[setting_key]:
            no_tetris_dataset_indexes = no_tetris_bundles[setting_key][dataset_key]

            # Insert e=0 data
            for method_name, method_indexes in df.loc[no_tetris_dataset_indexes].groupby("method").groups.items():
                # if "pearl" in method_name:
                #     continue

                method = {"name": method_labels[method_name]}
                max_k = df.loc[method_indexes]["k"].max()

                metrics_by_k_and_percentile = {
                    k: {
                        1: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.8: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.7: {"gen_throughputs": [], "vsrs": [], "latencies": []}
                    } for k in range(1, max_k + 1)
                }

                # Collect metrics for all bundles by k value
                for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                    for i, bundle in df.loc[indexes].iterrows():
                        # Print total times of preemption
                        if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                            print(
                                f"{YELLOW}[WARN]{RESET} "
                                f"{dataset}, target={target_model}, draft={draft_model}, "
                                f"bs={batch_size}, reqs={reqs}, "
                                f"method={method_name}, k={k}: "
                                f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                            )

                        for percentile in [1, 0.8, 0.7]:
                            bound = int(len(bundle['step_drafted_tokens']) * percentile)

                            gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                            vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                            e2el = bundle['total_latency'] / bundle['reqs']

                            # Store metrics for this bundle by k and percentile
                            metrics_by_k_and_percentile[k][percentile]["gen_throughputs"].append(gen_throughput)
                            metrics_by_k_and_percentile[k][percentile]["vsrs"].append(vsr)
                            metrics_by_k_and_percentile[k][percentile]["latencies"].append(e2el)

                metrics_by_percentile = {
                    percentile: {
                        "avg_gen_throughput": [np.nan] * max_k,
                        "std_gen_throughput": [np.nan] * max_k,
                        "avg_vsr": [np.nan] * max_k,
                        "std_vsr": [np.nan] * max_k,
                        "avg_latency": [np.nan] * max_k,
                        "std_latency": [np.nan] * max_k,
                    } for percentile in [1, 0.8, 0.7]
                }

                # Calculate means and stds
                for k in range(1, max_k + 1):
                    for percentile in [1, 0.8, 0.7]:
                        metrics = metrics_by_k_and_percentile[k][percentile]

                        # Skip if no data collected
                        if not metrics["gen_throughputs"]:
                            continue

                        # Calculate means
                        avg_gen = np.mean(metrics["gen_throughputs"])
                        avg_vsr = np.mean(metrics["vsrs"])
                        avg_latency = np.mean(metrics["latencies"])

                        # Calculate stds
                        std_gen = np.std(metrics["gen_throughputs"])
                        std_vsr = np.std(metrics["vsrs"])
                        std_latency = np.std(metrics["latencies"])

                        # Store back in the dictionary
                        metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                        metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                        metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                        metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                        metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                        metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

                trims = []
                for percentile in [1, 0.8, 0.7]:
                    trims.append({
                        "percentile": f"{int(percentile*100)}%",
                        **metrics_by_percentile[percentile]
                    })

                method["trims"] = trims
                methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, trim={percentile}"
            )
            report_minedraft_improvement(
                metrics_df.loc[percentile_indexes],
                targets,
                report_context
            )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_'
                            f'{percentile}_{target}'
                        ) + '.png'
                    ),
                    dpi=100
                )
                plt.close(fig)

### n = 1 & diff draft models

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency', 'vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12


for (target_model, batch_size, reqs), setting_indexes in df[(df['e'] == 0) & (df['n'] == 1) & (df['c'] <= 0) & (df['method'].isin(["psd", "sd"]))].groupby(
        ["target_model", "batch_size", "reqs"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        draft_models = {}

        for (draft_model_name, method), draft_model_indexes in df.loc[dataset_indexes].groupby(["draft_model", "method"]).groups.items():
            draft_model_key = (draft_model_name, method)
            draft_model = {"name": draft_model_name, "method": method}
            max_k = df.loc[draft_model_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[draft_model_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model_name}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method}, k={k}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            draft_model["trims"] = trims
            draft_models.update({draft_model_key: {"name": draft_model_name, "method": method, "trims": draft_model["trims"]}})

        # Prepare DataFrame for plotting
        rows = []
        for (draft_model_name, method), draft_model_data in draft_models.items():
            for trim in draft_model_data["trims"]:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'draft_model': draft_model_name,
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}
            
            # Assign colors to each draft model (same color for psd and sd of same draft model)
            unique_draft_models = metrics_df.loc[percentile_indexes, 'draft_model'].unique()
            color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
            draft_model_colors = {dm: color_cycle[i % len(color_cycle)] for i, dm in enumerate(unique_draft_models)}

            for (draft_model_name, method), draft_model_indexes in metrics_df.loc[percentile_indexes].groupby(['draft_model', 'method']).groups.items():
                grouped = metrics_df.loc[draft_model_indexes].groupby('k').first().dropna()
                
                # Set line style based on method: solid for psd, dotted for sd
                linestyle = '-' if method == 'psd' else ':'
                method_prefix = 'MineDraft' if method == 'psd' else 'Standard SD'
                color = draft_model_colors[draft_model_name]
                
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle=linestyle, 
                             label=f'{method_prefix} ({draft_model_name})', linewidth=linewidth, markersize=markersize, color=color)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2, color=color)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, "
                f"bs={batch_size}, reqs={reqs}, trim={percentile}"
            )
            print(report_context)

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_'
                            f'{reqs}_bs{batch_size}_'
                            f'{percentile}_{target}'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

### n = 1 & fixed c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

# Index e=0 data (without TETRIS)
no_tetris_bundles = {}
no_tetris_df = df.loc[(df['e'] == 0) & (df['n'] == 1)].copy()
no_tetris_df['c'] = no_tetris_df['batch_size'] * no_tetris_df['k']
for setting_group, setting_group_indexes in no_tetris_df.groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "c"]).groups.items():

    dataset_bundles = {}
    for dataset_name, dataset_indexes in df.loc[setting_group_indexes].groupby("dataset").groups.items():
        dataset_bundles[dataset_name] = dataset_indexes

    no_tetris_bundles[setting_group] = dataset_bundles


for (target_model, draft_model, batch_size, reqs, c), setting_indexes in df[(df['e'] > 0) & (df['n'] == 1) & (df['c'] > 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "c"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, e), method_indexes in df.loc[dataset_indexes].groupby(["method", "e"]).groups.items():
            # if method_name == "tetris":
            #     continue

            method = {"name": method_labels[method_name] + f' (extra={e})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), c={c}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Find matching e=0 data
        setting_key = (target_model, draft_model, batch_size, reqs, c)
        dataset_key = dataset_name
        if setting_key in no_tetris_bundles and dataset_key in no_tetris_bundles[setting_key]:
            no_tetris_dataset_indexes = no_tetris_bundles[setting_key][dataset_key]

            # Insert e=0 data
            for method_name, method_indexes in df.loc[no_tetris_dataset_indexes].groupby("method").groups.items():
                method = {"name": method_labels[method_name]}
                max_k = df.loc[method_indexes]["k"].max()

                metrics_by_k_and_percentile = {
                    k: {
                        1: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.8: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.7: {"gen_throughputs": [], "vsrs": [], "latencies": []}
                    } for k in range(1, max_k + 1)
                }

                # Collect metrics for all bundles by k value
                for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                    for i, bundle in df.loc[indexes].iterrows():
                        # Print total times of preemption
                        if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                            print(
                                f"{YELLOW}[WARN]{RESET} "
                                f"{dataset}, target={target_model}, draft={draft_model}, "
                                f"bs={batch_size}, reqs={reqs}, "
                                f"method={method_name}, k={k}, c={c}: "
                                f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                            )

                        for percentile in [1, 0.8, 0.7]:
                            bound = int(len(bundle['step_drafted_tokens']) * percentile)

                            gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                            vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                            e2el = bundle['total_latency'] / bundle['reqs']

                            # Store metrics for this bundle by k and percentile
                            metrics_by_k_and_percentile[k][percentile]["gen_throughputs"].append(gen_throughput)
                            metrics_by_k_and_percentile[k][percentile]["vsrs"].append(vsr)
                            metrics_by_k_and_percentile[k][percentile]["latencies"].append(e2el)

                metrics_by_percentile = {
                    percentile: {
                        "avg_gen_throughput": [np.nan] * max_k,
                        "std_gen_throughput": [np.nan] * max_k,
                        "avg_vsr": [np.nan] * max_k,
                        "std_vsr": [np.nan] * max_k,
                        "avg_latency": [np.nan] * max_k,
                        "std_latency": [np.nan] * max_k,
                    } for percentile in [1, 0.8, 0.7]
                }

                # Calculate means and stds
                for k in range(1, max_k + 1):
                    for percentile in [1, 0.8, 0.7]:
                        metrics = metrics_by_k_and_percentile[k][percentile]

                        # Skip if no data collected
                        if not metrics["gen_throughputs"]:
                            continue

                        # Calculate means
                        avg_gen = np.mean(metrics["gen_throughputs"])
                        avg_vsr = np.mean(metrics["vsrs"])
                        avg_latency = np.mean(metrics["latencies"])

                        # Calculate stds
                        std_gen = np.std(metrics["gen_throughputs"])
                        std_vsr = np.std(metrics["vsrs"])
                        std_latency = np.std(metrics["latencies"])

                        # Store back in the dictionary
                        metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                        metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                        metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                        metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                        metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                        metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

                trims = []
                for percentile in [1, 0.8, 0.7]:
                    trims.append({
                        "percentile": f"{int(percentile*100)}%",
                        **metrics_by_percentile[percentile]
                    })

                method["trims"] = trims
                methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                for target in targets:
                    fig, axs = plots[target]

                    if e == 0:
                        # Draw dotted horizontal line for non-tetris methods
                        axs.axhline(y=grouped[f'avg_{target}'].iloc[0], linestyle='--', label=method)
                        axs.fill_between(range(1, max_k+1),
                                         grouped[f'avg_{target}'] - grouped[f'std_{target}'],
                                         grouped[f'avg_{target}'] + grouped[f'std_{target}'],
                                         alpha=0.2)

                    else:
                        # Plot line with markers
                        axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                        axs.fill_between(grouped.index, 
                                         grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                         grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                         alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, c={c}, trim={percentile}"
            )
            report_minedraft_improvement(
                metrics_df.loc[percentile_indexes],
                targets,
                report_context
            )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_c{c}_'
                            f'{percentile}_{target}'
                        ) + '.png'
                    ),
                    dpi=100
                )
                plt.close(fig)

### n > 1 & default c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

# Index e=0 data (without TETRIS)
no_tetris_bundles = {}
for setting_group, setting_group_indexes in df[(df['e'] == 0) & (df['n'] > 1)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n"]).groups.items():

    dataset_bundles = {}
    for dataset_name, dataset_indexes in df.loc[setting_group_indexes].groupby("dataset").groups.items():
        dataset_bundles[dataset_name] = dataset_indexes

    no_tetris_bundles[setting_group] = dataset_bundles


for (target_model, draft_model, batch_size, reqs, n), setting_indexes in df[(df['e'] > 0) & (df['n'] > 1) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, e), method_indexes in df.loc[dataset_indexes].groupby(["method", "e"]).groups.items():
            # if "pearl" in method_name or e > 3 or method_name.startswith("tetris") and e > 2:
            #     continue

            method = {"name": method_labels[method_name] + f' (extra={e})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), n={n}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Find matching e=0 data
        setting_key = (target_model, draft_model, batch_size, reqs, n)
        dataset_key = dataset_name
        if setting_key in no_tetris_bundles and dataset_key in no_tetris_bundles[setting_key]:
            no_tetris_dataset_indexes = no_tetris_bundles[setting_key][dataset_key]

            # Insert e=0 data
            for method_name, method_indexes in df.loc[no_tetris_dataset_indexes].groupby("method").groups.items():
                # if "pearl" in method_name:
                #     continue

                method = {"name": method_labels[method_name]}
                max_k = df.loc[method_indexes]["k"].max()

                metrics_by_k_and_percentile = {
                    k: {
                        1: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.8: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.7: {"gen_throughputs": [], "vsrs": [], "latencies": []}
                    } for k in range(1, max_k + 1)
                }

                # Collect metrics for all bundles by k value
                for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                    for i, bundle in df.loc[indexes].iterrows():
                        # Print total times of preemption
                        if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                            print(
                                f"{YELLOW}[WARN]{RESET} "
                                f"{dataset}, target={target_model}, draft={draft_model}, "
                                f"bs={batch_size}, reqs={reqs}, "
                                f"method={method_name}, k={k}, n={n}: "
                                f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                            )

                        for percentile in [1, 0.8, 0.7]:
                            bound = int(len(bundle['step_drafted_tokens']) * percentile)

                            gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                            vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                            e2el = bundle['total_latency'] / bundle['reqs']

                            # Store metrics for this bundle by k and percentile
                            metrics_by_k_and_percentile[k][percentile]["gen_throughputs"].append(gen_throughput)
                            metrics_by_k_and_percentile[k][percentile]["vsrs"].append(vsr)
                            metrics_by_k_and_percentile[k][percentile]["latencies"].append(e2el)

                metrics_by_percentile = {
                    percentile: {
                        "avg_gen_throughput": [np.nan] * max_k,
                        "std_gen_throughput": [np.nan] * max_k,
                        "avg_vsr": [np.nan] * max_k,
                        "std_vsr": [np.nan] * max_k,
                        "avg_latency": [np.nan] * max_k,
                        "std_latency": [np.nan] * max_k,
                    } for percentile in [1, 0.8, 0.7]
                }

                # Calculate means and stds
                for k in range(1, max_k + 1):
                    for percentile in [1, 0.8, 0.7]:
                        metrics = metrics_by_k_and_percentile[k][percentile]

                        # Skip if no data collected
                        if not metrics["gen_throughputs"]:
                            continue

                        # Calculate means
                        avg_gen = np.mean(metrics["gen_throughputs"])
                        avg_vsr = np.mean(metrics["vsrs"])
                        avg_latency = np.mean(metrics["latencies"])

                        # Calculate stds
                        std_gen = np.std(metrics["gen_throughputs"])
                        std_vsr = np.std(metrics["vsrs"])
                        std_latency = np.std(metrics["latencies"])

                        # Store back in the dictionary
                        metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                        metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                        metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                        metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                        metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                        metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

                trims = []
                for percentile in [1, 0.8, 0.7]:
                    trims.append({
                        "percentile": f"{int(percentile*100)}%",
                        **metrics_by_percentile[percentile]
                    })

                method["trims"] = trims
                methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, n={n}, trim={percentile}"
            )
            print(report_context)
            # report_minedraft_improvement(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_n{n}_'
                            f'{percentile}_{target}'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

### n > 1 & diff draft models

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency', 'vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, batch_size, reqs, n), setting_indexes in df[(df['e'] == 0) & (df['n'] > 1) & (df['c'] <= 0) & (df['method'].isin(["psd", "sd"]))].groupby(
        ["target_model", "batch_size", "reqs", "n"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        draft_models = {}

        for (draft_model_name, method), draft_model_indexes in df.loc[dataset_indexes].groupby(["draft_model", "method"]).groups.items():
            draft_model_key = (draft_model_name, method)
            draft_model = {"name": draft_model_name, "method": method}
            max_k = df.loc[draft_model_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[draft_model_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model_name}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method}, k={k}, n={n}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            draft_model["trims"] = trims
            draft_models.update({draft_model_key: {"name": draft_model_name, "method": method, "trims": draft_model["trims"]}})

        # Prepare DataFrame for plotting
        rows = []
        for (draft_model_name, method), draft_model_data in draft_models.items():
            for trim in draft_model_data["trims"]:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'draft_model': draft_model_name,
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            # Assign colors to each draft model (same color for psd and sd of same draft model)
            unique_draft_models = metrics_df.loc[percentile_indexes, 'draft_model'].unique()
            color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
            draft_model_colors = {dm: color_cycle[i % len(color_cycle)] for i, dm in enumerate(unique_draft_models)}

            for (draft_model_name, method), draft_model_indexes in metrics_df.loc[percentile_indexes].groupby(['draft_model', 'method']).groups.items():
                grouped = metrics_df.loc[draft_model_indexes].groupby('k').first().dropna()

                # Set line style based on method: solid for psd, dotted for sd
                linestyle = '-' if method == 'psd' else ':'
                method_prefix = 'MineDraft' if method == 'psd' else 'Standard SD'
                color = draft_model_colors[draft_model_name]

                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle=linestyle,
                             label=f'{method_prefix} ({draft_model_name})', linewidth=linewidth, markersize=markersize, color=color)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2, color=color)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, "
                f"bs={batch_size}, reqs={reqs}, n={n}, trim={percentile}"
            )
            print(report_context)

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_'
                            f'{reqs}_bs{batch_size}_n{n}_'
                            f'{percentile}_{target}'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

### n > 1 & fixed c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

# Index e=0 data (without TETRIS)
no_tetris_bundles = {}
no_tetris_df = df.loc[(df['e'] == 0) & (df['n'] > 1)].copy()
no_tetris_df['c'] = no_tetris_df['batch_size'] * no_tetris_df['k']
for setting_group, setting_group_indexes in no_tetris_df.groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n", "c"]).groups.items():

    dataset_bundles = {}
    for dataset_name, dataset_indexes in df.loc[setting_group_indexes].groupby("dataset").groups.items():
        dataset_bundles[dataset_name] = dataset_indexes

    no_tetris_bundles[setting_group] = dataset_bundles


for (target_model, draft_model, batch_size, reqs, n, c), setting_indexes in df[(df['e'] > 0) & (df['n'] > 1) & (df['c'] > 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n", "c"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, e), method_indexes in df.loc[dataset_indexes].groupby(["method", "e"]).groups.items():
            # if method_name == "tetris":
            #     continue

            method = {"name": method_labels[method_name] + f' (extra={e})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), n={n}, c={c}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Find matching e=0 data
        setting_key = (target_model, draft_model, batch_size, reqs, n, c)
        dataset_key = dataset_name
        if setting_key in no_tetris_bundles and dataset_key in no_tetris_bundles[setting_key]:
            no_tetris_dataset_indexes = no_tetris_bundles[setting_key][dataset_key]

            # Insert e=0 data
            for method_name, method_indexes in df.loc[no_tetris_dataset_indexes].groupby("method").groups.items():
                method = {"name": method_labels[method_name]}
                max_k = df.loc[method_indexes]["k"].max()

                metrics_by_k_and_percentile = {
                    k: {
                        1: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.8: {"gen_throughputs": [], "vsrs": [], "latencies": []},
                        0.7: {"gen_throughputs": [], "vsrs": [], "latencies": []}
                    } for k in range(1, max_k + 1)
                }

                # Collect metrics for all bundles by k value
                for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                    for i, bundle in df.loc[indexes].iterrows():
                        # Print total times of preemption
                        if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                            print(
                                f"{YELLOW}[WARN]{RESET} "
                                f"{dataset}, target={target_model}, draft={draft_model}, "
                                f"bs={batch_size}, reqs={reqs}, "
                                f"method={method_name}, k={k}, n={n}, c={c}: "
                                f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                            )

                        for percentile in [1, 0.8, 0.7]:
                            bound = int(len(bundle['step_drafted_tokens']) * percentile)

                            gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                            vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                            e2el = bundle['total_latency'] / bundle['reqs']

                            # Store metrics for this bundle by k and percentile
                            metrics_by_k_and_percentile[k][percentile]["gen_throughputs"].append(gen_throughput)
                            metrics_by_k_and_percentile[k][percentile]["vsrs"].append(vsr)
                            metrics_by_k_and_percentile[k][percentile]["latencies"].append(e2el)

                metrics_by_percentile = {
                    percentile: {
                        "avg_gen_throughput": [np.nan] * max_k,
                        "std_gen_throughput": [np.nan] * max_k,
                        "avg_vsr": [np.nan] * max_k,
                        "std_vsr": [np.nan] * max_k,
                        "avg_latency": [np.nan] * max_k,
                        "std_latency": [np.nan] * max_k,
                    } for percentile in [1, 0.8, 0.7]
                }

                # Calculate means and stds
                for k in range(1, max_k + 1):
                    for percentile in [1, 0.8, 0.7]:
                        metrics = metrics_by_k_and_percentile[k][percentile]

                        # Skip if no data collected
                        if not metrics["gen_throughputs"]:
                            continue

                        # Calculate means
                        avg_gen = np.mean(metrics["gen_throughputs"])
                        avg_vsr = np.mean(metrics["vsrs"])
                        avg_latency = np.mean(metrics["latencies"])

                        # Calculate stds
                        std_gen = np.std(metrics["gen_throughputs"])
                        std_vsr = np.std(metrics["vsrs"])
                        std_latency = np.std(metrics["latencies"])

                        # Store back in the dictionary
                        metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                        metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                        metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                        metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                        metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                        metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

                trims = []
                for percentile in [1, 0.8, 0.7]:
                    trims.append({
                        "percentile": f"{int(percentile*100)}%",
                        **metrics_by_percentile[percentile]
                    })

                method["trims"] = trims
                methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                for target in targets:
                    fig, axs = plots[target]

                    if e == 0:
                        # Draw dotted horizontal line for non-tetris methods
                        axs.axhline(y=grouped[f'avg_{target}'].iloc[0], linestyle='--', label=method)
                        axs.fill_between(range(1, max_k+1),
                                         grouped[f'avg_{target}'].iloc[0] - grouped[f'std_{target}'].iloc[0],
                                         grouped[f'avg_{target}'].iloc[0] + grouped[f'std_{target}'].iloc[0],
                                         alpha=0.2)

                    else:
                        # Plot line with markers
                        axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                        axs.fill_between(grouped.index, 
                                         grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                         grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                         alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, c={c}, n={n}, trim={percentile}"
            )
            report_minedraft_improvement(
                metrics_df.loc[percentile_indexes],
                targets,
                report_context
            )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_c{c}_n{n}_'
                            f'{percentile}_{target}'
                        ) + '.png'
                    ),
                    dpi=100
                )
                plt.close(fig)

## Ablation Studies

### VSR vs _e_

#### n = 1 & default c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, draft_model, batch_size, reqs), setting_indexes in df[(df['e'] > 0) & (df['n'] == 1) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, k), method_indexes in df.loc[dataset_indexes].groupby(["method", "k"]).groups.items():
            if not method_name.startswith("ptetris"):
                continue

            method = {"name": method_labels[method_name] + f' (k={k})'}
            max_e = df.loc[method_indexes]["e"].max()

            metrics_by_e_and_percentile = {
                e: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for e in range(1, max_e + 1)
            }

            # Collect metrics for all bundles by e value
            for e, indexes in df.loc[method_indexes].groupby("e", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}): "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by e and percentile
                        metrics_by_e_and_percentile[e][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_e_and_percentile[e][percentile]["vsr"].append(vsr)
                        metrics_by_e_and_percentile[e][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_e,
                    "std_gen_throughput": [np.nan] * max_e,
                    "avg_vsr": [np.nan] * max_e,
                    "std_vsr": [np.nan] * max_e,
                    "avg_latency": [np.nan] * max_e,
                    "std_latency": [np.nan] * max_e,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for e in range(1, max_e + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_e_and_percentile[e][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][e-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][e-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][e-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][e-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][e-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][e-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for e, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'e': e,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        metrics_df = pd.DataFrame(rows).dropna()

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('e').first()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Extra Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, trim={percentile}"
            )
            print(report_context)
            # report_minedraft_improvement_vs_e(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            e_vals = np.sort(metrics_df.loc[percentile_indexes, 'e'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(e_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_'
                            f'{percentile}_{target}_vs_e'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

#### n = 1 & fixed c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, draft_model, batch_size, reqs, c), setting_indexes in df[(df['e'] > 0) & (df['n'] == 1) & (df['c'] > 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "c"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, k), method_indexes in df.loc[dataset_indexes].groupby(["method", "k"]).groups.items():
            if not method_name.startswith("ptetris"):
                continue

            method = {"name": method_labels[method_name] + f' (k={k})'}
            max_e = df.loc[method_indexes]["e"].max()

            metrics_by_e_and_percentile = {
                e: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for e in range(1, max_e + 1)
            }

            # Collect metrics for all bundles by e value
            for e, indexes in df.loc[method_indexes].groupby("e", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), c={c}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by e and percentile
                        metrics_by_e_and_percentile[e][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_e_and_percentile[e][percentile]["vsr"].append(vsr)
                        metrics_by_e_and_percentile[e][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_e,
                    "std_gen_throughput": [np.nan] * max_e,
                    "avg_vsr": [np.nan] * max_e,
                    "std_vsr": [np.nan] * max_e,
                    "avg_latency": [np.nan] * max_e,
                    "std_latency": [np.nan] * max_e,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for e in range(1, max_e + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_e_and_percentile[e][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][e-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][e-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][e-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][e-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][e-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][e-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for e, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'e': e,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        metrics_df = pd.DataFrame(rows).dropna()

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('e').first()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Extra Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, c={c}, trim={percentile}"
            )
            print(report_context)
            # report_minedraft_improvement_vs_e(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            e_vals = np.sort(metrics_df.loc[percentile_indexes, 'e'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(e_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_c{c}_'
                            f'{percentile}_{target}_vs_e'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

#### n > 1 & default c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, draft_model, batch_size, reqs, n), setting_indexes in df[(df['e'] > 0) & (df['n'] > 1) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, k), method_indexes in df.loc[dataset_indexes].groupby(["method", "k"]).groups.items():
            if not method_name.startswith("ptetris"):
                continue

            method = {"name": method_labels[method_name] + f' (k={k})'}
            max_e = df.loc[method_indexes]["e"].max()

            metrics_by_e_and_percentile = {
                e: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for e in range(1, max_e + 1)
            }

            # Collect metrics for all bundles by e value
            for e, indexes in df.loc[method_indexes].groupby("e", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), n={n}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by e and percentile
                        metrics_by_e_and_percentile[e][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_e_and_percentile[e][percentile]["vsr"].append(vsr)
                        metrics_by_e_and_percentile[e][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_e,
                    "std_gen_throughput": [np.nan] * max_e,
                    "avg_vsr": [np.nan] * max_e,
                    "std_vsr": [np.nan] * max_e,
                    "avg_latency": [np.nan] * max_e,
                    "std_latency": [np.nan] * max_e,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for e in range(1, max_e + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_e_and_percentile[e][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][e-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][e-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][e-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][e-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][e-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][e-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for e, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'e': e,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows).dropna()

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('e').first()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Extra Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, n={n}, trim={percentile}"
            )
            print(report_context)
            # report_minedraft_improvement_vs_e(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            e_vals = np.sort(metrics_df.loc[percentile_indexes, 'e'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(e_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_n{n}_'
                            f'{percentile}_{target}_vs_e'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

#### n > 1 & fixed c

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['vsr']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, draft_model, batch_size, reqs, n, c), setting_indexes in df[(df['e'] > 0) & (df['n'] > 1) & (df['c'] > 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs", "n", "c"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, k), method_indexes in df.loc[dataset_indexes].groupby(["method", "k"]).groups.items():
            if not method_name.startswith("ptetris"):
                continue

            method = {"name": method_labels[method_name] + f' (k={k})'}
            max_e = df.loc[method_indexes]["e"].max()

            metrics_by_e_and_percentile = {
                e: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for e in range(1, max_e + 1)
            }

            # Collect metrics for all bundles by e value
            for e, indexes in df.loc[method_indexes].groupby("e", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), n={n}, c={c}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by e and percentile
                        metrics_by_e_and_percentile[e][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_e_and_percentile[e][percentile]["vsr"].append(vsr)
                        metrics_by_e_and_percentile[e][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_e,
                    "std_gen_throughput": [np.nan] * max_e,
                    "avg_vsr": [np.nan] * max_e,
                    "std_vsr": [np.nan] * max_e,
                    "avg_latency": [np.nan] * max_e,
                    "std_latency": [np.nan] * max_e,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for e in range(1, max_e + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_e_and_percentile[e][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][e-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][e-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][e-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][e-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][e-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][e-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                for e, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'e': e,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        metrics_df = pd.DataFrame(rows).dropna()

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            for method, method_indexes in metrics_df.loc[percentile_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('e').first()
                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2)

                    # Add labels and styling
                    axs.set_xlabel('No. Extra Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, c={c}, n={n}, trim={percentile}"
            )
            print(report_context)
            # report_minedraft_improvement_vs_e(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            e_vals = np.sort(metrics_df.loc[percentile_indexes, 'e'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(e_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_c{c}_n{n}_'
                            f'{percentile}_{target}_vs_e'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)

### VSR v. Percentile

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}


for (target_model, draft_model, batch_size, reqs, n), setting_indexes in df[
    (df['e'] > 0)
    & (
        ((df['draft_model'] == 'Qwen3-0.6B') & (df['batch_size'] == 16) & (df['n'] <= 2))
        | ((df['draft_model'] == 'Llama-3.1-8B-Instruct') & (df['n'] == 2))
    )
    & (df['c'] <= 0)
].groupby(["target_model", "draft_model", "batch_size", "reqs", "n"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, k), method_indexes in df.loc[dataset_indexes].groupby(["method", "k"]).groups.items():
            if method_name not in ("ptetris", "tetris"):
                continue

            max_e = 5
            percentiles = [1.0, 0.98, 0.96, 0.94, 0.92, 0.9, 0.88, 0.86, 0.84, 0.82, 0.8, 0.78, 0.76, 0.74, 0.72, 0.7]

            vsr_by_e_and_percentile = {
                e: {
                    percentage: [] for percentage in percentiles
                } for e in range(1, max_e + 1)
            }

            # Collect metrics for all bundles by e value
            for e, indexes in df.loc[method_indexes].groupby("e", sort=True).groups.items():
                # if e > 2:
                #     break

                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k} + (extra={e}), n={n}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )
                    for percentile in percentiles:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100

                        # Store metrics for this bundle by e and percentile
                        vsr_by_e_and_percentile[e][percentile].append(vsr)

            vsr = {
                percentile: {
                    "avg": [np.nan] * max_e,
                    "std": [np.nan] * max_e,
                } for percentile in percentiles
            }

            # Calculate means and stds
            for e in range(1, max_e + 1):
                for percentile in percentiles:
                    metrics = vsr_by_e_and_percentile[e][percentile]

                    # Skip if no data collected
                    if not metrics:
                        continue

                    # Store back in the dictionary
                    vsr[percentile]["avg"][e-1] = np.mean(metrics)
                    vsr[percentile]["std"][e-1] = np.std(metrics)

            for percentile in percentiles:
                metrics = {
                    "percentile": int(percentile * 100),
                    "vsr": vsr[percentile]
                }
                methods.setdefault(method_labels[method_name] + f" (k={k})", []).append(metrics)

        # Prepare DataFrame for plotting
        rows = []
        for method, metrics_list in methods.items():
            for metrics in metrics_list:
                percentile = metrics['percentile']
                vsr = metrics['vsr']
                for e, (avg_vsr, std_vsr) in enumerate(
                    zip(vsr['avg'], vsr['std']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'percentile': percentile,
                        'e': e,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                    })


        metrics_df = pd.DataFrame(rows)

        for e, e_indexes in metrics_df.groupby('e').groups.items():
            plots = plt.subplots(figsize=(9, 9))

            for method, method_indexes in metrics_df.loc[e_indexes].groupby('method').groups.items():
                grouped = metrics_df.loc[method_indexes].groupby('percentile').first().dropna()

                fig, axs = plots

                # Plot line with markers
                axs.plot(grouped.index, grouped['avg_vsr'], marker='o', linestyle='-', label=method, linewidth=linewidth, markersize=markersize)
                axs.fill_between(grouped.index, 
                                    grouped['avg_vsr'] - grouped['std_vsr'], 
                                    grouped['avg_vsr'] + grouped['std_vsr'], 
                                    alpha=0.2)

                # Add labels and styling
                axs.set_xlabel('Percentile', fontsize=axis_label_fontsize)
                axs.set_ylabel('VSR', fontsize=axis_label_fontsize)
                axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                axs.legend(loc='upper left', fontsize=legend_fontsize)
                axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, n={n}, e={e}"
            )
            print(report_context)

            # Force integer x-ticks across subplots
            fig, axs = plots
            axs.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

            # Finalize plot
            fig.tight_layout()
            display(fig)
            fig.savefig(
                os.path.join(
                    plot_dir,
                    safename(
                        f'{dataset}_'
                        f'{target_model}_{draft_model}_'
                        f'{reqs}_bs{batch_size}_n{n}_'
                        f'e{e}_vsr_vs_percentage'
                    ) + '.png',
                ),
                dpi=100
            )
            plt.close(fig)

### Vary _m_

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 16
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

for (target_model, draft_model, reqs), setting_indexes in df[(df['e'] == 0) & (df['n'] == 1) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "reqs"]).groups.items():

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, batch_size), method_indexes in df.loc[dataset_indexes].groupby(["method", "batch_size"]).groups.items():
            if method_name not in ("psd", "sd"):
                continue

            method = {"name": method_labels[method_name] + f' (m={batch_size})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    "batch_size": batch_size,
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                batch_size = trim['batch_size']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'batch_size': batch_size,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            # Create a color map for m values - MineDraft and Standard SD with same m get same color
            unique_m_values = sorted(metrics_df.loc[percentile_indexes]['batch_size'].unique())
            color_palette = plt.cm.tab10.colors
            m_to_color = {m: color_palette[i % len(color_palette)] for i, m in enumerate(unique_m_values)}

            # Sort methods by batch_size (m) in increasing order
            method_to_batch = metrics_df.loc[percentile_indexes].groupby('method')['batch_size'].first().to_dict()
            methods_sorted = sorted(
                metrics_df.loc[percentile_indexes]['method'].unique(),
                key=lambda x: (method_to_batch[x], x.startswith('Standard'))
            )

            for method in methods_sorted:
                method_indexes = metrics_df.loc[percentile_indexes][metrics_df.loc[percentile_indexes]['method'] == method].index
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                
                # Get m value from batch_size column
                m_val = metrics_df.loc[method_indexes, 'batch_size'].iloc[0]
                color = m_to_color[m_val]
                
                # Standard SD uses dotted line, MineDraft uses solid line
                linestyle = ':' if method.startswith('Standard SD') else '-'

                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle=linestyle, color=color, label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2, color=color)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='upper left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, trim={percentile}"
            )
            print(report_context)
            # report_max_minedraft_improvement(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_'
                            f'{percentile}_{target}_vs_m'
                        ) + '.png'
                    ),
                    dpi=100
                )
                plt.close(fig)

### Vary _n_

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
from IPython.display import display

targets = ['gen_throughput', 'latency']
axis_label_fontsize = 28
legend_fontsize = 22
linewidth = 3
markersize = 12

method_labels = {
    'sd': 'Standard SD',
    'pearl_sd': 'PEARL',
    'psd': 'MineDraft (standalone)',
    'tetris': 'Tetris',
    'pearl_tetris': 'PEARL + Tetris',
    'ptetris': 'MineDraft',
    'eagle': 'EAGLE',
    'pearl_eagle': 'PEARL + EAGLE',
    'peagle': 'MineDraft + EAGLE',
    'eagle3': 'EAGLE-3',
    'pearl_eagle3': 'PEARL + EAGLE-3',
    'peagle3': 'MineDraft + EAGLE-3',
    'tetris_eagle': 'Tetris + EAGLE',
    'tetris_eagle3': 'Tetris + EAGLE-3',
    'ptetris_eagle': 'MineDraft + EAGLE',
    'pearl_tetris_eagle': 'PEARL + Tetris + EAGLE',
    'ptetris_eagle3': 'MineDraft + EAGLE-3',
    'pearl_tetris_eagle3': 'PEARL + Tetris + EAGLE-3',
}

for (target_model, draft_model, batch_size, reqs), setting_indexes in df[(df['e'] == 0) & (df['c'] <= 0)].groupby(
        ["target_model", "draft_model", "batch_size", "reqs"]).groups.items():
    if target_model == "Qwen3-32B" and (draft_model != "Qwen3-0.6B" or batch_size != 16) or target_model == "Meta-Llama-3.3-70B-Instruct-AWQ-INT4" and batch_size != 64:
        continue

    for dataset_name, dataset_indexes in df.loc[setting_indexes].groupby("dataset").groups.items():
        dataset = dataset_name if dataset_name == "ShareGPT" else "-".join(w.capitalize() for w in dataset_name.replace("_", "-").split("-"))
        methods = {}

        for (method_name, n), method_indexes in df.loc[dataset_indexes].groupby(["method", "n"]).groups.items():
            if method_name not in ("psd", "sd"):
                continue

            method = {"name": method_labels[method_name] + f' (n={n})'}
            max_k = df.loc[method_indexes]["k"].max()

            metrics_by_k_and_percentile = {
                k: {
                    1: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.8: {"gen_throughput": [], "vsr": [], "latency": []},
                    0.7: {"gen_throughput": [], "vsr": [], "latency": []}
                } for k in range(1, max_k + 1)
            }

            # Collect metrics for all bundles by k value
            for k, indexes in df.loc[method_indexes].groupby("k", sort=True).groups.items():
                for i, bundle in df.loc[indexes].iterrows():
                    # Print total times of preemption
                    if (preemptions := np.sum(bundle['step_preempted_requests'])) > 0:
                        print(
                            f"{YELLOW}[WARN]{RESET} "
                            f"{dataset}, target={target_model}, draft={draft_model}, "
                            f"bs={batch_size}, reqs={reqs}, "
                            f"method={method_name}, k={k}, n={n}: "
                            f"iter={i+1}, {BOLD}{YELLOW}Total Preemptions: {preemptions}{RESET}"
                        )

                    for percentile in [1, 0.8, 0.7]:
                        bound = int(len(bundle['step_drafted_tokens']) * percentile)

                        gen_throughput = np.sum(bundle['step_generated_tokens'][:bound]) / np.sum(bundle['step_generation_times'][:bound])
                        vsr = np.sum(bundle['step_accepted_tokens'][:bound]) / np.sum(bundle['step_verified_tokens'][:bound]) * 100
                        e2el = bundle['total_latency'] / bundle['reqs']

                        # Store metrics for this bundle by k and percentile
                        metrics_by_k_and_percentile[k][percentile]["gen_throughput"].append(gen_throughput)
                        metrics_by_k_and_percentile[k][percentile]["vsr"].append(vsr)
                        metrics_by_k_and_percentile[k][percentile]["latency"].append(e2el)

            metrics_by_percentile = {
                percentile: {
                    "avg_gen_throughput": [np.nan] * max_k,
                    "std_gen_throughput": [np.nan] * max_k,
                    "avg_vsr": [np.nan] * max_k,
                    "std_vsr": [np.nan] * max_k,
                    "avg_latency": [np.nan] * max_k,
                    "std_latency": [np.nan] * max_k,
                } for percentile in [1, 0.8, 0.7]
            }

            # Calculate means and stds
            for k in range(1, max_k + 1):
                for percentile in [1, 0.8, 0.7]:
                    metrics = metrics_by_k_and_percentile[k][percentile]

                    # Skip if no data collected
                    if not metrics["gen_throughput"]:
                        continue

                    # Calculate means
                    avg_gen = np.mean(metrics["gen_throughput"])
                    avg_vsr = np.mean(metrics["vsr"])
                    avg_latency = np.mean(metrics["latency"])

                    # Calculate stds
                    std_gen = np.std(metrics["gen_throughput"])
                    std_vsr = np.std(metrics["vsr"])
                    std_latency = np.std(metrics["latency"])

                    # Store back in the dictionary
                    metrics_by_percentile[percentile]["avg_gen_throughput"][k-1] = avg_gen
                    metrics_by_percentile[percentile]["std_gen_throughput"][k-1] = std_gen
                    metrics_by_percentile[percentile]["avg_vsr"][k-1] = avg_vsr
                    metrics_by_percentile[percentile]["std_vsr"][k-1] = std_vsr
                    metrics_by_percentile[percentile]["avg_latency"][k-1] = avg_latency
                    metrics_by_percentile[percentile]["std_latency"][k-1] = std_latency

            trims = []
            for percentile in [1, 0.8, 0.7]:
                trims.append({
                    "percentile": f"{int(percentile*100)}%",
                    "n": n,
                    **metrics_by_percentile[percentile]
                })

            method["trims"] = trims
            methods.update({method["name"]: method["trims"]})

        # Prepare DataFrame for plotting
        rows = []
        for method, trims in methods.items():
            for trim in trims:
                percentile = trim['percentile']
                n = trim['n']
                for k, (avg_gen_throughput, std_gen_throughput, avg_vsr, std_vsr, avg_latency, std_latency) in enumerate(
                    zip(trim['avg_gen_throughput'], trim['std_gen_throughput'],
                        trim['avg_vsr'], trim['std_vsr'],
                        trim['avg_latency'], trim['std_latency']),
                    1
                ):
                    rows.append({
                        'method': method,
                        'k': k,
                        'n': n,
                        'percentile': percentile,
                        'avg_gen_throughput': avg_gen_throughput,
                        'std_gen_throughput': std_gen_throughput,
                        'avg_vsr': avg_vsr,
                        'std_vsr': std_vsr,
                        'avg_latency': avg_latency,
                        'std_latency': std_latency,
                    })

        if not rows:
            continue

        metrics_df = pd.DataFrame(rows)

        for percentile, percentile_indexes in metrics_df.groupby('percentile').groups.items():
            plots = {target: plt.subplots(figsize=(9, 9)) for target in targets}

            # Create a color map for n values - MineDraft and Standard SD with same n get same color
            unique_n_values = sorted(metrics_df.loc[percentile_indexes]['n'].unique())
            color_palette = plt.cm.tab10.colors
            n_to_color = {n: color_palette[i % len(color_palette)] for i, n in enumerate(unique_n_values)}

            # Sort methods by n in increasing order
            method_to_n = metrics_df.loc[percentile_indexes].groupby('method')['n'].first().to_dict()
            methods_sorted = sorted(
                metrics_df.loc[percentile_indexes]['method'].unique(),
                key=lambda x: (method_to_n[x], x.startswith('Standard'))
            )

            for method in methods_sorted:
                method_indexes = metrics_df.loc[percentile_indexes][metrics_df.loc[percentile_indexes]['method'] == method].index
                grouped = metrics_df.loc[method_indexes].groupby('k').first().dropna()
                
                # Get n value from n column
                n_val = metrics_df.loc[method_indexes, 'n'].iloc[0]
                color = n_to_color[n_val]
                
                # Standard SD uses dotted line, MineDraft uses solid line
                linestyle = ':' if method.startswith('Standard SD') else '-'

                for target in targets:
                    fig, axs = plots[target]

                    # Plot line with markers
                    axs.plot(grouped.index, grouped[f'avg_{target}'], marker='o', linestyle=linestyle, color=color, label=method, linewidth=linewidth, markersize=markersize)
                    axs.fill_between(grouped.index, 
                                     grouped[f'avg_{target}'] - grouped[f'std_{target}'], 
                                     grouped[f'avg_{target}'] + grouped[f'std_{target}'], 
                                     alpha=0.2, color=color)

                    # Add labels and styling
                    axs.set_xlabel('No. Speculative Tokens', fontsize=axis_label_fontsize)
                    ylabels = {'gen_throughput': 'Throughput (tokens/s)', 'latency': 'E2E Latency (ms)', 'vsr': 'VSR'}
                    axs.set_ylabel(ylabels.get(target, target.capitalize()), fontsize=axis_label_fontsize)
                    axs.tick_params(axis='both', which='major', labelsize=axis_label_fontsize)
                    axs.legend(loc='center left' if target == 'latency' else 'lower left', fontsize=legend_fontsize)
                    axs.grid(True)

            report_context = (
                f"{dataset}, target={target_model}, draft={draft_model}, "
                f"bs={batch_size}, reqs={reqs}, trim={percentile}"
            )
            print(report_context)
            # report_max_minedraft_improvement(
            #     metrics_df.loc[percentile_indexes],
            #     targets,
            #     report_context
            # )

            # Force integer x-ticks across subplots
            k_vals = sorted(metrics_df.loc[percentile_indexes, 'k'].unique())
            for target, (fig, ax) in plots.items():
                ax.set_xticks(k_vals)
                ax.xaxis.set_major_locator(MaxNLocator(integer=True))
                ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))

                # Finalize plot
                fig.tight_layout()
                display(fig)
                fig.savefig(
                    os.path.join(
                        plot_dir,
                        safename(
                            f'{dataset}_'
                            f'{target_model}_{draft_model}_'
                            f'{reqs}_bs{batch_size}_'
                            f'{percentile}_{target}_vs_n'
                        ) + '.png',
                    ),
                    dpi=100
                )
                plt.close(fig)